# Money Laundering Detection
## Milestone 2 — Data Understanding & Cleaning

### Objective
The goal of this notebook is to inspect, clean, validate, and prepare the money laundering dataset for the next stages of the project.

### Dataset Overview & Data Dictionary
- **Total Records (Raw):** 100,500 rows × 12 columns
- **Target Variable:** `Is_laundering` (0 = Normal, 1 = Laundering)

| Column Name | Description | Data Type (Raw) | Corrected Type |
| :--- | :--- | :--- | :--- |
| `Time` | Transaction timestamp (time) | String | Datetime/Time |
| `Date` | Transaction date | String | Datetime |
| `Sender_account` | Unique ID of sending account | Float64 | String |
| `Receiver_account` | Unique ID of receiving account | Float64 | String |
| `Amount` | Transaction value | Float64 | Float64 |
| `Payment_currency` | Currency used for payment | String | String |
| `Received_currency` | Currency received | String | String |
| `Sender_bank_location` | Bank country/location of sender | String | String |
| `Receiver_bank_location` | Bank country/location of receiver | String | String |
| `Payment_type` | Transfer channel/type | String | String |
| `Is_laundering` | Target label (0 or 1) | Int64 | Categorical |
| `Laundering_type` | Specific pattern/type of laundering | String | Categorical |

In [13]:
import pandas as pd
from pathlib import Path

RAW_PATH = Path("data/money_laundering_dirty_for_cleaning.xlsx")
df = pd.read_excel(RAW_PATH)

df.head()

,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type
0,10:51:48,10/7/2022,3.884365e+09,3.678154e+09,92.61,Euros,UK pounds,UK,UK,Credit card,0,Normal_Small_Fan_Out
1,19:44:48,10/11/2022,7.900126e+09,3.009831e+09,5157.16,UK pounds,UK pounds,UK,Japan,Cross-border,1,Behavioural_Change_2
2,8:17:22,10/15/2022,8.175368e+09,3.447406e+09,7426.99,UK pounds,UK pounds,UAE,UK,Cheque,0,Normal_Fan_Out
3,15:04:00,10/18/2022,7.865536e+08,6.980070e+09,3391.18,UK pounds,UK pounds,US,Nigeria,ACH,0,Normal_Small_Fan_Out
4,4:48:34,10/29/2022,2.839583e+09,1.512033e+09,6951.29,US dollars,US dollars,UK,Germany,Debit card,0,Normal_Fan_In


In [14]:
print("Shape:", df.shape)
df.info()

Shape: (100500, 12)
<class 'pandas.DataFrame'>
RangeIndex: 100500 entries, 0 to 100499
Data columns (total 12 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Time                    100377 non-null  str    
 1   Date                    100343 non-null  str    
 2   Sender_account          100347 non-null  float64
 3   Receiver_account        100362 non-null  float64
 4   Amount                  100357 non-null  float64
 5   Payment_currency        100500 non-null  str    
 6   Received_currency       100500 non-null  str    
 7   Sender_bank_location    100500 non-null  str    
 8   Receiver_bank_location  100500 non-null  str    
 9   Payment_type            100346 non-null  str    
 10  Is_laundering           100500 non-null  int64  
 11  Laundering_type         100364 non-null  str    
dtypes: float64(3), int64(1), str(8)
memory usage: 15.5 MB


In [15]:
# 1. Missing Values Overview
missing_summary = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percentage (%)': (df.isnull().sum() / len(df)) * 100
})
print("=== MISSING VALUES SUMMARY ===")
print(missing_summary[missing_summary['Missing Count'] > 0])

# 2. Duplicate Rows Check
duplicate_count = df.duplicated().sum()
print(f"\n=== DUPLICATES CHECK ===")
print(f"Total Exact Duplicate Rows: {duplicate_count} ({(duplicate_count/len(df))*100:.2f}%)")

=== MISSING VALUES SUMMARY ===
                  Missing Count  Missing Percentage (%)
Time                        123                0.122388
Date                        157                0.156219
Sender_account              153                0.152239
Receiver_account            138                0.137313
Amount                      143                0.142289
Payment_type                154                0.153234
Laundering_type             136                0.135323

=== DUPLICATES CHECK ===
Total Exact Duplicate Rows: 2846 (2.83%)


In [16]:
# Decision: Convert Account IDs to String & Parse Date/Time
# Reason: Account IDs are categorical identifiers, not numerical features for arithmetic ops.

df_clean = df.copy()

# Fix Account IDs
df_clean['Sender_account'] = df_clean['Sender_account'].apply(lambda x: str(int(x)) if pd.notnull(x) else x)
df_clean['Receiver_account'] = df_clean['Receiver_account'].apply(lambda x: str(int(x)) if pd.notnull(x) else x)

# Fix Datatypes
df_clean['Date'] = pd.to_datetime(
    df_clean['Date'],
    errors='coerce'
)

df_clean['Time'] = pd.to_datetime(
    df_clean['Time'],
    format='%H:%M:%S',
    errors='coerce'
).dt.time

df_clean['Is_laundering'] = df_clean['Is_laundering'].astype('category')

print("Data types corrected successfully!")

Data types corrected successfully!


In [17]:
# Decision: Drop exact duplicates
# Reason: Exact duplicate rows across all attributes represent data logging errors.

before_dup = len(df_clean)
df_clean = df_clean.drop_duplicates()
after_dup = len(df_clean)

print(f"Dropped {before_dup - after_dup} exact duplicate rows.")

Dropped 2846 exact duplicate rows.


In [18]:
# Decision: Fill Missing Categoricals with explicit 'Unknown' categories
# Reason: Avoid dropping rows unnecessarily when missingness represents unrecorded categorical data.

df_clean['Payment_type'] = df_clean['Payment_type'].fillna('Unknown')
df_clean['Laundering_type'] = df_clean['Laundering_type'].fillna('None/Unknown')

# Note on Laundering_type:
# Retained in clean dataset for traceability/analysis, but MUST NOT be used 
# as a predictive feature in modeling (Target Leakage).

# Drop minimal missing values in critical identifiers
df_clean = df_clean.dropna(subset=['Sender_account', 'Receiver_account', 'Amount', 'Date'])

print("Missing values treated. Remaining missing count:")
print(df_clean.isnull().sum())

Missing values treated. Remaining missing count:
Time                      123
Date                        0
Sender_account              0
Receiver_account            0
Amount                      0
Payment_currency            0
Received_currency           0
Sender_bank_location        0
Receiver_bank_location      0
Payment_type                0
Is_laundering               0
Laundering_type             0
dtype: int64


In [19]:
# Decision: Investigate outliers in 'Amount' but keep them in the dataset.
# Reason: Extreme values in transaction amounts are realistic in financial systems 
# and often represent key indicators of money laundering activities.

print("=== AMOUNT OUTLIERS INVESTIGATION ===")
print(df_clean['Amount'].describe())

# Check extreme quantiles
print("\n99th Percentile:", df_clean['Amount'].quantile(0.99))
print("Max Value:", df_clean['Amount'].max())

=== AMOUNT OUTLIERS INVESTIGATION ===
count    9.706600e+04
mean     1.569287e+04
std      1.688050e+05
min      1.000000e-02
25%      2.303605e+03
50%      5.911295e+03
75%      1.007747e+04
max      6.213932e+06
Name: Amount, dtype: float64

99th Percentile: 105674.69
Max Value: 6213931.56


In [20]:
# Before vs After Comparison Summary Table
summary_data = {
    'Metric': ['Total Rows', 'Columns', 'Duplicate Rows', 'Missing Values (Total)', 'Sender_account Dtype'],
    'Before Cleaning': [df.shape[0], df.shape[1], duplicate_count, df.isnull().sum().sum(), str(df['Sender_account'].dtype)],
    'After Cleaning': [df_clean.shape[0], df_clean.shape[1], df_clean.duplicated().sum(), df_clean.isnull().sum().sum(), str(df_clean['Sender_account'].dtype)]
}

comparison_df = pd.DataFrame(summary_data)
print("=== BEFORE vs AFTER CLEANING SUMMARY ===")
print(comparison_df)

=== BEFORE vs AFTER CLEANING SUMMARY ===
                   Metric Before Cleaning After Cleaning
0              Total Rows          100500          97066
1                 Columns              12             12
2          Duplicate Rows            2846              0
3  Missing Values (Total)            1004            123
4    Sender_account Dtype         float64            str


In [21]:
# Save clean dataset
CLEAN_PATH = Path("data/money_laundering_clean.csv")
df_clean.to_csv(CLEAN_PATH, index=False)
print(f"Clean dataset saved successfully to {CLEAN_PATH}")

Clean dataset saved successfully to data\money_laundering_clean.csv
